# 第 10 章 ニューラルネットワーク

直線では分けられない XOR を、隠れ層を積むことで解きます。学習は誤差逆伝播法で行います。

対応する記事: [第 10 章 ニューラルネットワーク（Polyglot Notebook（F#） の言語版）](../../../docs/article/grokking-machine-learning/fsharp/ch10.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch10NeuralNetworks.fs"

open GrokkingMl.Ch10NeuralNetworks

## XOR は直線で分けられない

対角線上の 2 点が同じクラスなので、**1 本の直線では絶対に分けられません。**

In [2]:
let points: Point list = [ [ 0.0; 0.0 ]; [ 0.0; 1.0 ]; [ 1.0; 0.0 ]; [ 1.0; 1.0 ] ]
let labels = [ 0; 1; 1; 0 ]

List.zip points labels
|> List.iter (fun (point, label) -> printfn "(%.0f, %.0f) → %d" point[0] point[1] label)

(

0

, 

0

) → 

...

1

, 

1

) → 

0

## 隠れ層の幅を変えて比べる

**隠れ層があっても、ニューロンが 1 つでは足りません。** 実質「直線を 1 本引いてから変換する」だけなので、表現力はロジスティック回帰と変わらないからです。

**「層を足せば強くなる」ではなく「十分な幅の隠れ層が要る」** ということです。

In [3]:
for hidden in [ 1; 2; 4 ] do
    let m, losses = trainWith hidden 0.5 20000 0 points labels
    printfn "隠れ層 %d ニューロン  正解率 %.2f  損失 %.4f → %.4f" hidden
        (accuracy m points labels) (List.head losses) (List.last losses)

隠れ層 

1

 ニューロン  正解率 

0.75

  損失 

...

1.00

  損失 

0.7560

 → 

0.0008

## 学習後の予測

隠れ層 4 ニューロンなら、**4 点すべてを 0.999 以上の確信で当てられます。**

In [4]:
let model, losses = trainWith 4 0.5 20000 0 points labels

List.zip points labels
|> List.iter (fun (point, label) ->
    printfn "(%.0f, %.0f) 正解=%d  予測確率 %.4f" point[0] point[1] label
        (predictProbability model point))

(

0

, 

0

) 正解=

...

1

) 正解=

0

  予測確率 

0.0011

## 勾配消失

シグモイドの微分は最大 0.25、両端では 0 に近づきます。**層を深く積むとこの小さな値が掛け合わされ、入力側の層がほとんど学習しなくなります。**

現代のネットワークが ReLU を使う理由がここにあります。

In [5]:
printfn "%8s %10s" "出力" "微分"

for output in [ 0.001; 0.1; 0.5; 0.9; 0.999 ] do
    printfn "%8.3f %10.6f" output (sigmoidDerivative output)

printfn ""
printfn "10 層積んだときの積 %.2e" (0.25 ** 10.0)

      出力

        微分

   0.001

  0.000999

   0.100

...

  0.090000

   0.999

  0.000999

10 層積んだときの積 

9.54e-007

## 試してみる: 学習の途中経過

損失がどう下がるかを見ます。**XOR は最初しばらく停滞してから、あるところで急に解けます。**

In [6]:
for epoch in 0..2000..19999 do
    let value = losses[epoch]
    let bar = String.replicate (int (value * 50.0)) "#"
    printfn "epoch %6d  損失 %.4f  %s" epoch value bar

epoch 

     0

  損失 

0.7560

#####################################

...

0.0011

epoch 

 18000

  損失 

0.0010